# Strategic Plan -> Top-3 Funding Calls (Phase 1)

This notebook implements the Phase 1 baseline: for each strategic plan, retrieve candidate chunks from Chroma, aggregate at funding-call level, and return a top-3 ranking.


In [ ]:
%pip install -q chromadb pypdf sentence-transformers pandas tqdm


In [ ]:
from pathlib import Path
import re
from typing import Dict, List

import chromadb
import pandas as pd
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from tqdm.auto import tqdm


In [ ]:
PROJECT_ROOT = Path.cwd()
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

STRATEGIC_PLAN_DIR = PROJECT_ROOT / "docs" / "strategicplans"
EXPECTED_SP_FILES = {
    "Naples_Federico_Piano_strategico_2021_2026.pdf",
    "Politechnico_di_Milano_2023-2025.pdf",
    "Piano Strategico Luiss 2021-2024 (per sito).pdf",
    "Bocconi_Piano_Strategico2021-2025&Vision2030.pdf",
    "LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf",
    "Sapienza_2021_2027.pdf",
}

N_RESULTS = 30
TOP_K_CALLS = 3
EVIDENCE_PER_CALL = 3
STRONG_HIT_THRESHOLD = 0.55

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma dir: {CHROMA_DIR}")
print(f"Collection: {COLLECTION_NAME}")
print(f"SP directory: {STRATEGIC_PLAN_DIR}")


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_sp_text(pdf_path: Path) -> Dict:
    reader = PdfReader(str(pdf_path))
    page_texts: List[str] = []

    for page in reader.pages:
        text = clean_text(page.extract_text() or "")
        if len(text) >= 40:
            page_texts.append(text)

    return {
        "sp_id": pdf_path.stem,
        "source_file": pdf_path.name,
        "text": "\n".join(page_texts),
        "non_empty_pages": len(page_texts),
    }


sp_paths = sorted(STRATEGIC_PLAN_DIR.glob("*.pdf"))
found = {p.name for p in sp_paths}
missing = EXPECTED_SP_FILES - found
if missing:
    raise FileNotFoundError(f"Missing strategic plan files: {sorted(missing)}")

strategic_plans = []
for path in tqdm(sp_paths, desc="Loading strategic plans"):
    if path.name in EXPECTED_SP_FILES:
        strategic_plans.append(extract_sp_text(path))

print(f"Loaded {len(strategic_plans)} strategic plans")
for sp in strategic_plans:
    print(f" - {sp['source_file']}: non_empty_pages={sp['non_empty_pages']}, chars={len(sp['text'])}")


In [ ]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embed_fn)
print("Chroma collection is ready")


In [ ]:
def similarity_from_distance(distance: float) -> float:
    return 1.0 / (1.0 + float(distance))


def rank_calls_for_sp(sp: Dict, n_results: int = N_RESULTS) -> Dict:
    result = collection.query(
        query_texts=[sp['text']],
        n_results=n_results,
        include=['metadatas', 'documents', 'distances'],
    )

    metadatas = result['metadatas'][0]
    documents = result['documents'][0]
    distances = result['distances'][0]

    by_call: Dict[str, Dict] = {}

    for meta, doc, dist in zip(metadatas, documents, distances):
        call = meta.get('source_file', 'unknown')
        page = meta.get('page', None)
        sim = similarity_from_distance(dist)

        if call not in by_call:
            by_call[call] = {
                'call_name': call,
                'similarities': [],
                'pages': set(),
                'evidence': [],
            }

        by_call[call]['similarities'].append(sim)
        if page is not None:
            by_call[call]['pages'].add(page)
        by_call[call]['evidence'].append({
            'page': page,
            'similarity': round(sim, 4),
            'chunk': doc[:600],
        })

    ranked = []
    for call, payload in by_call.items():
        sims = sorted(payload['similarities'], reverse=True)
        top_sims = sims[:5]

        semantic_score = sum(top_sims) / max(1, len(top_sims))
        coverage_score = min(1.0, len(payload['pages']) / 6.0)
        strong_hits = sum(1 for s in sims if s >= STRONG_HIT_THRESHOLD)
        consistency_score = min(1.0, strong_hits / 5.0)
        final_score = 0.7 * semantic_score + 0.2 * coverage_score + 0.1 * consistency_score

        ranked.append({
            'call_name': call,
            'semantic_score': round(semantic_score, 4),
            'coverage_score': round(coverage_score, 4),
            'consistency_score': round(consistency_score, 4),
            'final_score': final_score,
            'evidence': sorted(payload['evidence'], key=lambda x: x['similarity'], reverse=True)[:EVIDENCE_PER_CALL],
        })

    ranked = sorted(ranked, key=lambda x: x['final_score'], reverse=True)

    for row in ranked:
        row['final_score'] = round(row['final_score'], 4)

    return {
        'sp_id': sp['sp_id'],
        'sp_file': sp['source_file'],
        'top_calls': ranked[:TOP_K_CALLS],
        'raw_hits': len(metadatas),
    }


In [ ]:
phase1_results = []
for sp in tqdm(strategic_plans, desc='Ranking funding calls'):
    phase1_results.append(rank_calls_for_sp(sp))

rows = []
for result in phase1_results:
    for idx, call in enumerate(result['top_calls'], start=1):
        rows.append({
            'sp_file': result['sp_file'],
            'rank': idx,
            'call_name': call['call_name'],
            'final_score': call['final_score'],
            'semantic_score': call['semantic_score'],
            'coverage_score': call['coverage_score'],
            'consistency_score': call['consistency_score'],
        })

ranking_df = pd.DataFrame(rows).sort_values(['sp_file', 'rank']).reset_index(drop=True)
ranking_df


In [ ]:
def print_evidence_for_sp(sp_file: str):
    matches = [x for x in phase1_results if x['sp_file'] == sp_file]
    if not matches:
        print(f'No result for {sp_file}')
        return

    result = matches[0]
    print(f"Strategic plan: {result['sp_file']}")
    for i, call in enumerate(result['top_calls'], start=1):
        print(f"\nRank {i}: {call['call_name']} | final_score={call['final_score']}")
        for ev in call['evidence']:
            page_label = ev['page'] if ev['page'] is not None else 'n/a'
            print(f" - page={page_label}, sim={ev['similarity']}: {ev['chunk'][:180]}...")

# Example:
print_evidence_for_sp(phase1_results[0]['sp_file'])
